# 🤝 Módulo 11 — Orchestration: Handoff

> **Objetivo:** Construir un sistema de atención multi-agente donde el control se transfiere dinámicamente entre especialistas.

📚 **Doc oficial:** [Handoff Orchestration (Python)](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/handoff?pivots=programming-language-python)

## ¿Qué es Handoff Orchestration?

Los agentes están conectados en **topología de malla** — cada uno puede transferir el control
a otro agente especializado según el contexto. No hay un orquestador central que decida.

```
        ┌──────────┐
        │  triage  │
        └──┬─────┬─┘
       ┌───┘     └───┐
  ┌────▼──┐      ┌──▼─────┐
  │ refund │     │ orders │
  └────────┘     └────────┘
```

Ideal para:
- **Customer support** — el agente correcto atiende según la consulta
- Sistemas expertos con dominios bien delimitados
- Cualquier escenario que requiera delegación dinámica

## Handoff vs Agent-as-Tools

| Aspecto | Handoff | Agent-as-Tools |
|---------|---------|----------------|
| Control | Se transfiere completamente | El agente principal retiene el control |
| Ownership | El receptor asume la tarea | El principal sigue siendo responsable |
| Contexto | Conversación completa se transfiere | Sólo lo necesario se pasa al tool |

## API clave

```python
from agent_framework.orchestrations import HandoffBuilder, HandoffAgentUserRequest

workflow = (
    HandoffBuilder(name="support", participants=[triage, refund, order])
    .with_start_agent(triage)
    .build()
)
```


## ⚙️ Setup

In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados.")


## 1️⃣ Definir tools y agentes especializados

Cada agente tiene tools relevantes a su dominio. El `triage_agent` no tiene tools — sólo enruta.


In [ ]:
from typing import Annotated
from agent_framework import tool


@tool(approval_mode="never_require")
def process_refund(
    order_number: Annotated[str, "Order number to process refund for"],
) -> str:
    """Simulated function to process a refund for a given order number."""
    return f"Refund processed successfully for order {order_number}."


@tool(approval_mode="never_require")
def check_order_status(
    order_number: Annotated[str, "Order number to check status for"],
) -> str:
    """Simulated function to check the status of a given order number."""
    return f"Order {order_number} is currently being processed and will ship in 2 business days."


@tool(approval_mode="never_require")
def process_return(
    order_number: Annotated[str, "Order number to process return for"],
) -> str:
    """Simulated function to process a return for a given order number."""
    return f"Return initiated for order {order_number}. Return instructions sent via email."


client = create_chat_client()

triage_agent = Agent(
    client=client,
    name="triage_agent",
    description="Triage agent that handles general inquiries.",
    instructions=(
        "You are frontline support triage. Route customer issues to the appropriate "
        "specialist agents based on the problem described."
    ),
)

refund_agent = Agent(
    client=client,
    name="refund_agent",
    description="Agent that handles refund requests.",
    instructions="You process refund requests.",
    tools=[process_refund],
)

order_agent = Agent(
    client=client,
    name="order_agent",
    description="Agent that handles order tracking and shipping.",
    instructions="You handle order and shipping inquiries.",
    tools=[check_order_status],
)

return_agent = Agent(
    client=client,
    name="return_agent",
    description="Agent that handles return processing.",
    instructions="You manage product return requests.",
    tools=[process_return],
)

print("✅ 4 agentes especializados creados")


## 2️⃣ Construir el workflow con reglas de handoff

Por defecto **todos los agentes pueden delegar a todos**. Usa `add_handoff(from, [to_list])`
para restringir las rutas.

> ℹ️ Aun con reglas custom, todos los agentes siguen conectados (malla) para que compartan
> contexto. Las reglas solo controlan **quién puede iniciar el handoff**.


In [ ]:
from agent_framework.orchestrations import HandoffBuilder

workflow = (
    HandoffBuilder(
        name="customer_support_handoff",
        participants=[triage_agent, refund_agent, order_agent, return_agent],
    )
    .with_start_agent(triage_agent)  # triage recibe el input inicial
    # Triage NO puede mandar directo a refund — primero debe pasar por return
    .add_handoff(triage_agent, [order_agent, return_agent])
    # Solo el agente de return puede delegar a refund (devoluciones → reembolso)
    .add_handoff(return_agent, [refund_agent])
    # Todos los especialistas pueden volver a triage
    .add_handoff(order_agent, [triage_agent])
    .add_handoff(return_agent, [triage_agent])
    .add_handoff(refund_agent, [triage_agent])
    .build()
)

print("✅ Workflow de handoff construido con rutas restringidas")


## 3️⃣ Modo autónomo (sin input humano)

Por defecto el handoff es **interactivo** — si un agente decide NO delegar, el workflow pausa
y pide input al usuario. Esto es porque un handoff se dispara con un tool call especial:
si el agente sólo responde texto, no se sabe qué hacer después.

Para demos no interactivos (y notebooks ✨), activa `with_autonomous_mode()` — el framework
inyecta una respuesta sintética automáticamente para que la conversación avance.


In [ ]:
# Modo autónomo: el framework auto-responde "continúa" si un agente no delega.
# Para el demo limitamos a 0 turnos extra (basta con la primera ronda de handoff).
workflow = (
    HandoffBuilder(
        name="autonomous_support",
        participants=[triage_agent, refund_agent, order_agent, return_agent],
    )
    .with_start_agent(triage_agent)
    .with_autonomous_mode(turn_limits={triage_agent.name: 0})
    .build()
)

from agent_framework import AgentResponseUpdate

last_author = None
async for event in workflow.run("Process a refund for order 12345", stream=True):
    if event.type in ("output", "intermediate") and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        if update.author_name != last_author:
            if last_author is not None:
                print()
            print(f"[{update.author_name}]: {update.text}", end="", flush=True)
            last_author = update.author_name
        else:
            print(update.text, end="", flush=True)
print("\n\n✅ Demo de modo autónomo completado.")


## 4️⃣ Modo interactivo (HITL)

Para producción típicamente quieres que el workflow pause y pida input real del usuario.
El patrón es:

1. Iniciar con `workflow.run_stream("...")`
2. Detectar eventos `event.type == "request_info"` con `isinstance(event.data, HandoffAgentUserRequest)`
3. Recolectar respuestas del usuario y reenviarlas con `workflow.run(responses={...})`

> 💡 En este notebook simulamos el "input del usuario" con respuestas predefinidas para que
> sea ejecutable end-to-end sin intervención manual.


In [ ]:
from agent_framework.orchestrations import HandoffAgentUserRequest

# El demo interactivo "real" requiere input humano en cada turno.
# Aquí mostramos el PATRÓN; cópialo a tu app y reemplaza `next(user_replies)`
# por `input(...)` o por la respuesta de tu UI.

workflow = (
    HandoffBuilder(
        name="interactive_support",
        participants=[triage_agent, refund_agent, order_agent, return_agent],
    )
    .with_start_agent(triage_agent)
    .build()
)

print("✅ Workflow interactivo construido (4 agentes en topología de malla).")
print("   Para probarlo interactivamente, descomenta y ejecuta este código:")
print()
print("   # async for event in workflow.run('I need help', stream=True):")
print("   #     if event.type == 'request_info' and isinstance(event.data, HandoffAgentUserRequest):")
print("   #         user_input = input('Tú: ')")
print("   #         responses = {event.request_id: HandoffAgentUserRequest.create_response(user_input)}")
print("   #         async for e in workflow.run(responses=responses, stream=True):")
print("   #             ...  # procesar eventos")

# Pequeña demo no-interactiva: enviar terminate() inmediatamente
pending = []
async for event in workflow.run("Hello", stream=True):
    if event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
        pending.append(event)
        break  # Sólo necesitamos detectar el primer request

if pending:
    responses = {pending[0].request_id: HandoffAgentUserRequest.terminate()}
    async for _ in workflow.run(responses=responses, stream=True):
        pass

print("\n✅ Patrón HITL demostrado (workflow terminado limpiamente).")


## 🎯 Resumen

| Capacidad | API |
|-----------|-----|
| Workflow básico | `HandoffBuilder(name=..., participants=[...]).with_start_agent(triage).build()` |
| Restringir rutas | `.add_handoff(from_agent, [to_agents])` |
| Modo autónomo | `.with_autonomous_mode(turn_limits={...})` |
| Modo interactivo | Detectar `request_info` + `HandoffAgentUserRequest` |
| Responder al request | `HandoffAgentUserRequest.create_response("...")` |
| Terminar workflow | `HandoffAgentUserRequest.terminate()` |

> 📖 Para casos con tool approval mezclado, revisa la sección "Tool Approval in Handoff Workflows" de la [doc oficial](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/handoff?pivots=programming-language-python#advanced-tool-approval-in-handoff-workflows).

**Siguiente módulo →** [Módulo 12 — Orchestration Group Chat](./12_orchestration_groupchat.ipynb)
